# NHS England Maternal Care Pathways Master Pipeline
## Stage 6 - Merge cleaned MSDS vars, diagnoses/comorbidity vars, and unit/hospital busyness vars and perform final filtering

### Compiled by Ethan Phillips (Oxford)

### Last updated 17.12.2025

In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR
import tempfile
import os

%run "/Workspace/Shared/Helper_Methods_EP"


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name_1 = "all_datasets_diagnoses_stage"
read_table_name_2 = "msds_with_readmission_stage"
read_table_name_3 = "msds_ld_data"

df_all_diags = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_1}")
df_readmit = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_2}")
df_ld = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name_3}")

In [0]:
df_master = (
    df_readmit.drop(*["ld_hosp_start_date"])
    .join(df_all_diags, on="UniqPregID", how="left")
    .join(df_ld, on="UniqPregID", how="left")
)

In [0]:
df_master = df_master.withColumn(
    "ld_hosp_stay_length",
    when((col("ld_hosp_start_date").isNotNull()) | (col("ld_hosp_disch_date").isNotNull()), datediff(col("ld_hosp_disch_date"), col("ld_hosp_start_date"))).otherwise(None)
).withColumn(
    "long_ld_hosp_los",
    when(col("ld_hosp_stay_length").isNotNull() & (col("ld_hosp_stay_length")>=4.0), 1).when(col("ld_hosp_stay_length").isNotNull() & (col("ld_hosp_stay_length")<4.0), 0).otherwise(None)
)

In [0]:
w = Window.partitionBy("Person_ID_DEID").orderBy("last_period_date").rowsBetween(Window.unboundedPreceding, -1)

df_master = (
    df_master.
    withColumn(
        "previous_pregnancy",
        F.when(F.count("*").over(w) >= 1, 1).otherwise(0)
    )
    .withColumn(
        "previous_pregnancy",
        F.when(((col("previous_pregnancy") == 1) | (col("prior_live_birth") == 1) | (col("prior_csection") == 1) | (col("prior_24w_loss") == 1) | (col("prior_stillbirth") == 1)), 1).otherwise(0)
    )
)

In [0]:
#display(df_master.select("UniqPregID", "num_births"))
print(df_master.filter(col("num_births") == 0).count())
print(df_master.filter(col("num_births") == 1).count())
print(df_master.filter(col("num_births") >= 2).count())
print(df_master.filter(col("num_births").isNull()).count())

In [0]:
print(f"Number of pregnancies before filtering for number of births: {df_master.count()}")

df_master = df_master.filter(
    (col("num_births") == 0) | (col("num_births") == 1)
)

print(f"Number of pregnancies after filtering for number of births: {df_master.count()}")


In [0]:
print(f"Number of pregnancies before filtering for birth year >=2021 : {df_master.count()}")

df_master = df_master.filter(
    (col("birth_year").isNotNull()) &
    (col("birth_year") >= 2021)
).drop(*["birth_2018", "birth_2019", "birth_2020"])

print(f"Number of pregnancies after filtering for birth year >=2021 : {df_master.count()}")

In [0]:
print(f"Number of pregnancies before filtering for non-null IMD : {df_master.count()}")

df_master = df_master.filter(
    (col("imd_decile").isNotNull())
)

print(f"Number of pregnancies after filtering for non-null IMD : {df_master.count()}")

In [0]:
print(f"Number of pregnancies before filtering for birth year +/- 1yr of LD admission year : {df_master.count()}")

df_master = (
    df_master
    .filter(
        (col("ld_hosp_start_date").isNull()) | (F.abs(col("birth_year") - F.year(col("ld_hosp_start_date"))) <= 1) 
    )
)

print(f"Number of pregnancies after filtering for birth year +/- 1yr of LD admission year : {df_master.count()}")

In [0]:
print(f"Missingness in hospital site: {(df_master.filter(col("ld_hosp_org_site_id").isNull()).count())/(df_master.count())}")

df_master = df_master.withColumn('ld_hosp_org_site_id', when(col('ld_hosp_org_site_id').isNull(), 'ZZZZ').otherwise(col('ld_hosp_org_site_id')))

print(f"Missingness in hospital site after replacing Nulls: {(df_master.filter(col("ld_hosp_org_site_id").isNull()).count())/(df_master.count())}")

In [0]:
df_master = df_master.drop(
    *[col for col in df_master.columns if (
        col.lower().startswith("diagnosiscode") or \
        col.lower().startswith("diag2") or \
        col.lower().startswith("diag3") or \
        col.lower().startswith("diag4")
    )]
)

In [0]:
df_master.columns

In [0]:
print(df_master.count())

In [0]:
write_table_name_1 = "msds_diag_busy_filtered_final_stage"

df_master.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name_1}")

In [0]:
df_master = df_master.repartition(40, "birth_year", "imd_decile")

# Generate list of (birth_year, imd_decile) tuples present in df_master
year_decile_pairs = (
    df_master
    .select("birth_year", "imd_decile")
    .distinct()
    .collect()
)

part_ids = [(row["birth_year"], row["imd_decile"]) for row in year_decile_pairs]

In [0]:
partition_file_name = "pre_impute_data_partition_171125"
impute_partition_file_name = "post_impute_data_partition_171125"

part_tabs = []
for tup in part_ids:
    if tup[1] is None:
        temp_df = df_master.filter((col("birth_year") == tup[0]) & (col("imd_decile").isNull()))
    else:
        temp_df = df_master.filter((col("birth_year") == tup[0]) & (col("imd_decile") == tup[1]))
    
    temp_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{partition_file_name}_yr{tup[0]}_imd{tup[1]}")

In [0]:
%skip
def print_basic_stats(df):
    # Compute total number of rows once
    total_rows = df.count()

    # Collect statistics for each column
    stats = []
    for col_name in [name for name, dtype in df.dtypes if dtype in ('int', 'float', 'double')]:
        agg_vals = (
            df
            .agg(
                F.avg(F.col(col_name)).alias("avg_val"),
                (F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)) / F.lit(total_rows)).alias("pct_null")
            )
            .collect()[0]
        )
        stats.append(
            (col_name, agg_vals["avg_val"], agg_vals["pct_null"])
        )

    # Create a DataFrame with the results and display it
    result_df = spark.createDataFrame(
        stats,
        schema=["variable", "average", "percent_nulls"]
    )

    display(result_df)

print_basic_stats(df_master)


In [0]:
outcome_cols = [
    'ld_disch_dest', 
    'neonatal_crit_incident',
    'neonatal_critical_care',
    'maternal_crit_incident',
    'very_low_birth_weight',
    'low_birth_weight',
    'any_low_birth_weight',
    'high_birth_weight',
    'normal_birth_weight',
    'normal_apgar',
    'low_apgar',
    'severe_distress_apgar',
    'moderate_distress_apgar',
    'outcome_live_birth',
    'outcome_antepartum_still_birth',
    'outcome_intrapartum_still_birth',
    'outcome_unknown_timing_still_birth',
    'outcome_any_still_birth',
    'extremely_preterm_birth',
    'very_preterm_birth',
    'moderate_to_late_preterm_birth',
    'not_preterm_birth',
    'preterm_birth_before_32w',
    'delivery_spont_vertex',
    'delivery_other_ceph',
    'delivery_low_forcep_not_breech',
    'delivery_other_forcep_not_breech',
    'delivery_ventouse_vac',
    'delivery_breech',
    'delivery_breech_extract',
    'delivery_elect_csection',
    'delivery_emergency_csection',
    'delivery_not_listed',
    'fetus_presentation_cephalic',
    'fetus_presentation_breech',
    'fetus_presentation_transverse',
    'emergency_readmission',
    'ld_hosp_stay_length',
    'long_ld_hosp_los'
]

outcome_cols = outcome_cols + [col for col in df_master.columns if col.endswith("_this_pregnancy")]

In [0]:
ref_cols = [
    'person_id_deid',
    'epikey',
    'fyear',
    'last_period_date',
    'antenatal_appt_date',
    'booking_org',
    'ccg_residence',
    'matserv_disch_date',
    'est_delivery_date',
    'labour_onset_date',
    'ld_site_id',
    'ld_disch_date',
    'birth_day_of_week',
    'birth_month',
    'birth_am_or_pm',
    'DiagDescriptions_PrePreg_ALL',
    'DiagDescriptions_DuringPreg_ALL',
    'ld_hosp_org_site_setting_id',
    'ld_hosp_org_site_id',
    'ld_hosp_start_date',
    'ld_hosp_disch_date'
]

In [0]:
try:
    partition_file_name
except:
    partition_file_name = "pre_impute_data_partition_171125"
    impute_partition_file_name = "post_impute_data_partition_171125"

for tup in part_ids: 
    part_df = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{partition_file_name}_yr{tup[0]}_imd{tup[1]}")

    print(f"Year = {tup[0]}, IMD decile = {tup[1]}")
    n = part_df.count()
    if n == 0:
        print("No patients. Skipping...")
        continue
    print(f"Number of patients: {n}")

    reference = part_df.select(['UniqPregID']+ref_cols)
    print(reference)
    part_df = part_df.drop(*ref_cols)
    print(part_df)
    new_df = knn_impute(spark, part_df, k=5, outcome_cols=outcome_cols)

    new_df = new_df.join(reference, on="UniqPregID", how="left")

    new_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{impute_partition_file_name}_yr{tup[0]}_imd{tup[1]}")

In [0]:
table_prefix = "post_impute_data_partition_171125"

# List all tables in the target database/schema
db_name = "dsa_712819_x8g2j.dsa_712819_x8g2j_collab"
tables = [
    t.name
    for t in spark.catalog.listTables(db_name)
    if t.name.startswith(table_prefix)
]

# Load each table and union them into a single DataFrame
df_combined = None
for tbl in tables:
    df = spark.read.table(f"{db_name}.{tbl}")
    df_combined = df if df_combined is None else df_combined.unionByName(df, allowMissingColumns=True)

print(f"Number of patients before imputation: {df_master.count()}")
print(f"Number of patients after imputation: {df_combined.count()}")
      
# Save combined file
df_combined.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.post_impute_data_combined_191125")


In [0]:
df_master_imputed = df_combined
print(f"Missingness in support_ind: {df_master_imputed.filter(col("support_ind").isNull()).count()/df_master_imputed.count()}")

In [0]:
write_table_name_2 = "msds_diag_busy_filtered_final_imputed_stage"

df_master_imputed.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name_2}")